In [3]:
import os
from dotenv import load_dotenv

ARIZE_SPACE_ID=os.getenv("ARIZE_SPACE_ID")
ARIZE_API_KEY =os.getenv("ARIZE_API_KEY")
TAVILY_API_KEY =os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

In [4]:
from arize.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.instrumentation.agno import AgnoInstrumentor
import httpx

model_id= "E-Commerce Parallel Execuation"
tracer_provider=register(
    space_id=os.getenv("ARIZE_SPACE_ID"),
    api_key=os.getenv("ARIZE_API_KEY"),
    project_name=model_id,
    set_global_tracer_provider=True
)

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
AgnoInstrumentor().instrument(tracer_provider=tracer_provider)

OpenTelemetry Tracing Details
|  Arize Project: E-Commerce Parallel Execuation
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
from opentelemetry import trace
tracer=trace.get_tracer(__name__)


In [6]:
def _search_api(query: str)-> str | None:
    tavily_api=os.getenv("TAVILY_API_KEY")
    if not tavily_api:
        return None
    try:
        resp=httpx.post(
            "https://api.tavily.com/search",
            json={
                "api_key":tavily_api,
                "query":query,
                "max_result":3,
                "search_depth":"basic",
                "include_answer":True,
            },
            timeout=8,
        )
        data=resp.json()
        answer=data.get("answer") or ""
        snippets=[r.get("content","") for r in data.get("result",[])]
        combined=" ".join([answer] + snippets).strip()
        return combined[:400] if combined else None
    except Exception:
        return None
    
def _compact(text: str, limit: int=200)-> str:
    cleaned=" ".join(text.split())
    if len(cleaned) <= limit:
        return cleaned
    else:
        return cleaned[:limit].rsplit(" ",1)[0]


In [7]:
from agno.tools import tool

@tool
def product_search(product: str)-> str:
    q=f"{product} features price specification and reviews"
    s=_search_api(q)
    if s:
        return f"{product} info: {_compact(s)}"
    return f"{product} has various models with different features and prices."

@tool 
def product_compare(product1: str, product2: str)-> str:
    q=f"{product1} vs {product2} comparison, features, price, pros and cons"
    s=_search_api(q)
    if s:
        return f"comparison {product1} vs {product2}: {_compact(s)}"
    return f"{product1} and {product2} differ in features, performance and priceing "

@tool 
def budget_finder(product: str, budget: str)-> str:
    q=f"best {product} under {budget} recommendation"
    s=_search_api(q)
    if s:
        return f"Best {product} under {budget}: {_compact(s)}"
    return f"you can find good {product} option under {budget} with balanced features."        



In [8]:
import asyncio
from agno.agent import Agent
from agno.models.openai import OpenAIChat



In [13]:
# Define Subagent
product_search_agent=Agent(
    name="SearchInfo",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[product_search],
    instructions=["Search for and return a concise, formatted list of the top 3-5 products matching the user specific price, brand, and feature requirement using the product search tool"]
)

product_compare_agent=Agent(
    name="CompareInfo",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[product_compare],
    instructions=["Analyze and compare the provided product side by side, highlighting key difference in price, features, and value to recommend the best option for the user."]
)

budget_finder_agent=Agent(
    name="BudgetInfo",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[budget_finder],
    instructions=["find the most cost-effective options that strictly adhere to the user specified budget, highlighting any potential hidden cost or available discount."]
)

synthesizer = Agent(
    name="Synthesizer",
    model=OpenAIChat(id="gpt-4o-mini"),
    description="You combine information from multiple subagents into a final, cohesive response.",
    instructions=[
        "You are an expert product recommender.",
        "Review the information provided by the search, budget, and comparison subagents.",
        "Synthesize their findings into a single, seamless, and easy-to-read recommendation.",
        "Do not mention the subagents directly to the user."
    ],
    markdown=True,
)


In [14]:
from wrapt.decorators import synchronized
from opentelemetry import trace
from openinference.semconv.trace import SpanAttributes
tracer=trace.get_tracer(__name__)

# Run Subagent concurrently and synthesize
async def search_and_compare_products(product_query: str, budget: str, preferences: str):
    with tracer.start_as_current_span("parallelizationAgent") as span:
        span.set_attribute(SpanAttributes.OPENINFERENCE_SPAN_KIND, "agent")
        span.set_attribute("product_query", product_query)
        span.set_attribute("budget",budget)
        span.set_attribute("preferences", preferences)

        # Run all three agent concurrently
        # Each agent recieves the specific input it need to perfoam is independant task
        search_task=product_search_agent.arun(product_query)
        budget_task=budget_finder_agent.arun(budget)
        compare_task=product_compare_agent.arun(preferences)

        # wait for all three subagent to finish their teaks

        search_info, budget_info, compare_info=await asyncio.gather(search_task, budget_task, compare_task)
        # combine result via one final llm call
        final_prompt=f"""
        combine the following into a cohesive, well-structured product
        recommendation for : {product_query}. keep the response helfull, objectives
        and easy to read and under 1000 words.

        [Product Search Results]
        {search_info}

        [Budget options and Constraints]
        {budget_info}

        [Feature comparison and Analysis]
        {compare_info}
        """
        # Assuming you have an agent named 'synthesizer to bring it all together
        final_recommendation=await synthesizer.arun(final_prompt)
        return final_recommendation


In [15]:
product_query="noise cancelling headphones"
budget="under $300"
preferences="prioritize battery life and comfort"

final=await search_and_compare_products(product_query=product_query, budget=budget, preferences=preferences)
print(final)